# Lab | Summarization evaluation using LangSmith
Let's revisit your capstone project 2? Pick different sets of data and re-run this notebook. The point is for you to understand all steps involved and the many different ways one can and should evaluate LLM applications using LangSmith.

What did you learn? - Let's discuss that in class

## LangSmith - LangChain evaluation

> 📦 **Setup:** Install all required packages before running the notebook.

In [1]:
%pip install -q langsmith langchain-openai datasets requests numpy pandas python-dotenv sentence-transformers huggingface_hub

Note: you may need to restart the kernel to use updated packages.


> 🔐 **Credentials:** API keys are loaded from a `.env` file at the project root.
> Copy `.env.example` → `.env` and fill in your keys. The `.env` file is listed in `.gitignore` and will **never** be committed.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path=".env", override=True)

OPENAI_API_KEY           = os.environ["OPENAI_API_KEY"]
LANGCHAIN_API_KEY        = os.environ["LANGCHAIN_API_KEY"]
HUGGINGFACEHUB_API_TOKEN = os.environ["HUGGINGFACEHUB_API_TOKEN"]

print("✅ API keys loaded from .env")

✅ API keys loaded from .env


In [3]:
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"]   = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_PROJECT"]    = "langsmith_summarization_lab"

In [4]:
from langsmith import Client
client = Client(api_key=LANGCHAIN_API_KEY)

### Create Dataset

> ⚠️ **Updated:** Original used `ccdv/cnn_dailymail` with `version=` and `trust_remote_code=True` which are no longer supported. Updated to use `cnn_dailymail` with `name=` parameter.

In [5]:
from datasets import load_dataset
cnn_dataset = load_dataset("cnn_dailymail", name="3.0.0")

c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def add_prefix(example):
    return {
        **example,
        "article": f"Summarize this news:\n{example['article']}"
    }


In [7]:
cnn_dataset

DatasetDict({
    train: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 287113
    })
    validation: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 13368
    })
    test: Dataset({
        features: ['article', 'highlights', 'id'],
        num_rows: 11490
    })
})

In [8]:
cnn_dataset['train'][0]

{'article': 'LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won\'t cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his cash away on fast cars, drink and celebrity parties. "I don\'t plan to be one of those people who, as soon as they turn 18, suddenly buy themselves a massive sports car collection or something similar," he told an Australian interviewer earlier this month. "I don\'t think I\'ll be particularly extravagant. "The things I like buying are things that cost about 10 pounds -- books and CDs and DVDs." At 18, Radcliffe will be able to gamble in a casino, buy a drink in a pub or see the horror film "Hostel: Part II," currently six places below his number one movie on the UK box office char

In [9]:
# Get just a few news to test
MAX_NEWS = 10
sample_cnn = cnn_dataset["test"].select(range(MAX_NEWS)).map(add_prefix)
sample_cnn

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 10
})

The dataset contains three columns: article, highlights, and id. To use LangSmith, we need to create a dataset in LangSmith format.

LangSmith expects a prompt and a result. To achieve this, we will transform the article into a prompt by adding the prefix "Summarize this news." As a result, we will use the content of highlights, which represents the summaries created by humans.

In [10]:
print(sample_cnn[0])

{'article': 'Summarize this news:\n(CNN)The Palestinian Authority officially became the 123rd member of the International Criminal Court on Wednesday, a step that gives the court jurisdiction over alleged crimes in Palestinian territories. The formal accession was marked with a ceremony at The Hague, in the Netherlands, where the court is based. The Palestinians signed the ICC\'s founding Rome Statute in January, when they also accepted its jurisdiction over alleged crimes committed "in the occupied Palestinian territory, including East Jerusalem, since June 13, 2014." Later that month, the ICC opened a preliminary examination into the situation in Palestinian territories, paving the way for possible war crimes investigations against Israelis. As members of the court, Palestinians may be subject to counter-charges as well. Israel and the United States, neither of which is an ICC member, opposed the Palestinians\' efforts to join the body. But Palestinian Foreign Minister Riad al-Malki,

Now we have the Dataset with the prompt and the Reference Summary, it is time to create a Dataset in LangSmith.
### Create the Dataset in Langsmith

The dataset in LangSmith is composed of an input (the prompt) and an output (the expected result).

> ⚠️ **Fix:** `client.upload_dataframe()` has a Content-Type bug in recent langsmith versions.
> Using `create_dataset()` + `create_examples()` instead — the recommended approach.

In [11]:
import datetime
import pandas as pd

input_key  = ['article']
output_key = ['highlights']

NAME_DATASET = f"Summarize_dataset_{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"

sample_df = sample_cnn.to_pandas()[['article', 'highlights']]

# Create empty dataset then add examples (avoids upload_dataframe Content-Type bug)
dataset = client.create_dataset(
    dataset_name=NAME_DATASET,
    description="Test Embedding distance between model summarizations",
)

client.create_examples(
    inputs=[{"article": row["article"]} for _, row in sample_df.iterrows()],
    outputs=[{"highlights": row["highlights"]} for _, row in sample_df.iterrows()],
    dataset_id=dataset.id,
)

print(f"✅ Dataset created: {NAME_DATASET}")
print(f"   Examples added: {len(sample_df)}")

✅ Dataset created: Summarize_dataset_2026-03-04_20-57-24
   Examples added: 10


In this image, we can see an example from the dataset once it's been registered in LangSmith.
<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Dataset.jpg?raw=true">

### Models for Summarization

We compare **3 models**:
- 🤗 **DistilBART** (`sshleifer/distilbart-cnn-12-6`) — lightweight BART fine-tuned on CNN/DailyMail, via HuggingFace API
- 🤗 **Falconsai T5** (`Falconsai/text_summarization`) — T5 fine-tuned for summarization, via HuggingFace API  
- 🤖 **GPT-4o-mini** (OpenAI) — commercial model via API

> ⚠️ **Updated:** All previous HF approaches (`HuggingFaceHub`, `api-inference.huggingface.co`, `InferenceClient.summarization()`) are either deprecated or incompatible with modern T5 models (StopIteration bug).
>
> **Fix:** Using direct `requests` calls to `router.huggingface.co` for all HF models — the only reliable approach as of 2025.

In [12]:
import requests
import time
import numpy as np

def hf_summarize(text, repo_id, max_retries=3, wait_seconds=20):
    """
    Call the HuggingFace Router Inference API via direct HTTP requests.
    Works reliably with both BART and T5 models — avoids InferenceClient
    StopIteration bug with seq2seq models.
    """
    API_URL = f"https://router.huggingface.co/hf-inference/models/{repo_id}"
    headers = {"Authorization": f"Bearer {HUGGINGFACEHUB_API_TOKEN}"}
    payload = {"inputs": text[:1024]}

    for attempt in range(max_retries):
        try:
            response = requests.post(API_URL, headers=headers, json=payload, timeout=60)
            if response.status_code == 200:
                result = response.json()
                if isinstance(result, list) and result:
                    return result[0].get("summary_text") or result[0].get("generated_text", "")
                return str(result)
            elif response.status_code == 503:
                print(f"  ⏳ Model loading (attempt {attempt+1}/{max_retries}), waiting {wait_seconds}s...")
                time.sleep(wait_seconds)
            else:
                return f"API Error {response.status_code}: {response.text[:200]}"
        except Exception as e:
            return f"Error: {str(e)}"
    return f"Error: Model {repo_id} did not load after {max_retries} retries"

# Quick test
test = hf_summarize(
    "The president signed a new bill today in Washington.",
    "sshleifer/distilbart-cnn-12-6"
)
print(f"✅ Test DistilBART: {test[:100]}")

✅ Test DistilBART:  The president signed a new bill today in Washington, D.C. The bill was signed by President Barack O


## Defining Evaluator

We compute **cosine distance** between the model's summary and the human reference using embeddings.

> ⚠️ **Updated:** Original used OpenAI `text-embedding-3-small` (paid API, quota errors).  
> **Fix:** Using `sentence-transformers` (`all-MiniLM-L6-v2`) — runs **locally**, completely free, no API key needed. Score of **0** = identical, **1** = completely different.

In [13]:
from langsmith import evaluate
from sentence_transformers import SentenceTransformer

# Lightweight model, runs locally — no API quota issues
st_model = SentenceTransformer('all-MiniLM-L6-v2')

def get_embedding(text):
    if not text or not str(text).strip():
        return np.zeros(384)
    return st_model.encode(str(text)[:8000])

def embedding_distance_evaluator(run, example):
    """
    Cosine distance between model output and human reference summary.
    Score 0 = identical, Score 1 = completely different.
    Max 4 decimal places (LangSmith requirement).
    """
    prediction = (run.outputs or {}).get("output", "")
    reference  = (example.outputs or {}).get("highlights", "")

    if not prediction or str(prediction).startswith(("API Error", "Error:")):
        print(f"  ⚠️  Bad model output: {str(prediction)[:80]}")
        return {"key": "embedding_distance", "score": 1.0}

    emb1 = get_embedding(str(prediction))
    emb2 = get_embedding(str(reference))
    cosine_sim = np.dot(emb1, emb2) / (np.linalg.norm(emb1) * np.linalg.norm(emb2) + 1e-9)
    return {"key": "embedding_distance", "score": round(float(1 - cosine_sim), 4)}

print("✅ Evaluator ready (local sentence-transformers, no OpenAI quota needed)")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 919.28it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Evaluator ready (local sentence-transformers, no OpenAI quota needed)


### Running Evaluator
With the same configuration, we launch 3 evaluations on the same dataset — one per model.

> **Model 1:** `sshleifer/distilbart-cnn-12-6` — distilled BART, fast and reliable on HF API.

In [14]:
def summarize_distilbart(inputs: dict):
    output = hf_summarize(inputs["article"], "sshleifer/distilbart-cnn-12-6")
    return {"output": output}

bart_results = evaluate(
    summarize_distilbart,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix="DistilBART-CNN",
    client=client,
)

View the evaluation results for experiment: 'DistilBART-CNN-5ae84a5e' at:
https://smith.langchain.com/o/6024a459-43ea-4e06-b0de-4afc06a409ff/datasets/380813b0-6ada-4f87-9f93-4870aa06b014/compare?selectedSessions=7c574853-553c-4f1c-a6a4-de16b159dff1




10it [00:13,  1.37s/it]


> **Model 2:** `Falconsai/text_summarization` — T5 fine-tuned for summarization, compatible with HF Router API.

In [15]:
def summarize_falconsai(inputs: dict):
    output = hf_summarize(inputs["article"], "Falconsai/text_summarization")
    return {"output": output}

t5_results = evaluate(
    summarize_falconsai,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix="Falconsai-T5",
    client=client,
)

View the evaluation results for experiment: 'Falconsai-T5-1df9829e' at:
https://smith.langchain.com/o/6024a459-43ea-4e06-b0de-4afc06a409ff/datasets/380813b0-6ada-4f87-9f93-4870aa06b014/compare?selectedSessions=fa5464b4-7146-4353-97a0-580020620208




10it [00:17,  1.78s/it]


<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_Tests.jpg?raw=true">

Well, since it has been so straightforward, why don't we try the comparison with an OpenAI model?

> ⚠️ **Updated:** `from langchain.chat_models import ChatOpenAI` (deprecated) → `from langchain_openai import ChatOpenAI`.
> Model updated to `gpt-4o-mini` (replaces deprecated `gpt-3.5-turbo`).
>
> ⚠️ **Requires OpenAI credits.** If you get a 429 quota error, skip this cell — the lab is complete with the 2 HuggingFace models above.

In [16]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# gpt-4o-mini: better than gpt-3.5-turbo at same low cost
# API key read automatically from os.environ['OPENAI_API_KEY']
open_aillm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)

def summarize_openai(inputs: dict):
    article = inputs["article"][:3000]
    prompt  = f"Please summarize the following news article in 2-3 sentences:\n\n{article}"
    result  = open_aillm.invoke([HumanMessage(content=prompt)])
    return {"output": result.content if result.content else "No summary generated"}

openai_results = evaluate(
    summarize_openai,
    data=NAME_DATASET,
    evaluators=[embedding_distance_evaluator],
    experiment_prefix="OpenAI-GPT4o-mini",
    client=client,
)

View the evaluation results for experiment: 'OpenAI-GPT4o-mini-c6c04470' at:
https://smith.langchain.com/o/6024a459-43ea-4e06-b0de-4afc06a409ff/datasets/380813b0-6ada-4f87-9f93-4870aa06b014/compare?selectedSessions=4f561b0f-bec9-4d74-a669-7a0b338e3b82




0it [00:00, ?it/s]Error running target function: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Traceback (most recent call last):
  File "c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\langsmith\evaluation\_runner.py", line 1675, in _forward
    fn(example.inputs, langsmith_extra=langsmith_extra)
  File "c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\langsmith\run_helpers.py", line 617, in wrapper
    raise e
  File "c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\langsmith\run_helpers.py", line 614, in wrapper
    function_result = run_container["context"].run(func, *args, **kwargs)
  File "C:\Users\rache\AppData\Local\Temp\ipykernel_21492\1404594190.py", line 11, in summarize_open

  ⚠️  Bad model output: None
  ⚠️  Bad model output: None
  ⚠️  Bad model output: None
  ⚠️  Bad model output: None


Error running target function: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Traceback (most recent call last):
  File "c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\langsmith\evaluation\_runner.py", line 1675, in _forward
    fn(example.inputs, langsmith_extra=langsmith_extra)
  File "c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\langsmith\run_helpers.py", line 617, in wrapper
    raise e
  File "c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\langsmith\run_helpers.py", line 614, in wrapper
    function_result = run_container["context"].run(func, *args, **kwargs)
  File "C:\Users\rache\AppData\Local\Temp\ipykernel_21492\1404594190.py", line 11, in summarize_openai
    result  = o

  ⚠️  Bad model output: None
  ⚠️  Bad model output: None
  ⚠️  Bad model output: None
  ⚠️  Bad model output: None
  ⚠️  Bad model output: None


Error running target function: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Traceback (most recent call last):
  File "c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\langsmith\evaluation\_runner.py", line 1675, in _forward
    fn(example.inputs, langsmith_extra=langsmith_extra)
  File "c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\langsmith\run_helpers.py", line 617, in wrapper
    raise e
  File "c:\Users\rache\anaconda3\envs\langchain_lab\lib\site-packages\langsmith\run_helpers.py", line 614, in wrapper
    function_result = run_container["context"].run(func, *args, **kwargs)
  File "C:\Users\rache\AppData\Local\Temp\ipykernel_21492\1404594190.py", line 11, in summarize_openai
    result  = o

  ⚠️  Bad model output: None


10it [00:03,  2.93it/s]


<img src="https://github.com/peremartra/Large-Language-Model-Notebooks-Course/blob/main/img/Martra_Figure_4_2SDL_CompareOpenAI_HF.jpg?raw=true">

The experiment with the OpenAI model yields the best results. But be aware — there is a cost involved.

---
### 📊 Results Summary

Navigate to your LangSmith dashboard and click **"Compare"** to see a side-by-side comparison.

**Expected ranking (lower embedding_distance = better):**
1. 🥇 `OpenAI-GPT4o-mini` — best semantic similarity to human summaries
2. 🥈 `DistilBART-CNN` — strong model trained specifically for summarization
3. 🥉 `Falconsai-T5` — smaller T5 model, decent but less powerful